# Turbofan simulation analysis

Run three C++ cases and compare the results with selected observations from NASA TM 84903. The station plots and transient use the synthetic reference engine. The PW-100 comparison labels the flight points, historical deck, and synthetic pressure-ratio proxy separately.

Build the simulator first, then select **Restart Kernel and Run All Cells**. See the [notebook guide](../docs/notebook-analysis.md) for installation and the commands behind each plot.

In [ ]:
import os
import subprocess
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

%matplotlib inline

cwd = Path.cwd()
ROOT = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "include" / "turbofan" / "cycle.hpp").is_file()
     and (path / "CMakeLists.txt").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the simulator checkout.")

# Edit these tuples to use custom cases, maps, or a control profile.
DRY_COMMAND = ("design",)
WET_COMMAND = ("wet",)
TRANSIENT_COMMAND = ("controlled",)
TRACE_STATION = "nozzle_inlet"

def repo_path(path):
    resolved = Path(path).expanduser()
    return resolved if resolved.is_absolute() else ROOT / resolved

CLI = repo_path(os.environ.get("TURBOFAN_CLI", "build/turbofan-cli"))
OUTPUT = repo_path(os.environ.get("TURBOFAN_ANALYSIS_DIR", "build/analysis"))
if not CLI.is_file() or not os.access(CLI, os.X_OK):
    raise FileNotFoundError(
        f"Executable not found at {CLI}. Build with CMake or set TURBOFAN_CLI."
    )
OUTPUT.mkdir(parents=True, exist_ok=True)

DRY_COLOR = "#23628B"
WET_COLOR = "#B54C1E"
DECK_COLOR = "#58616E"
plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
})

def run_csv(stem, command, required_columns):
    args = [str(CLI), *command]
    completed = subprocess.run(args, cwd=ROOT, capture_output=True, text=True)
    (OUTPUT / f"{stem}.log").write_text(completed.stderr, encoding="utf-8")
    if completed.returncode != 0:
        raise RuntimeError(f"{' '.join(args)} exited with status {completed.returncode}\n{completed.stderr}")
    frame = pd.read_csv(StringIO(completed.stdout))
    missing = set(required_columns) - set(frame)
    if missing or frame.empty:
        raise ValueError(f"{stem} produced no usable rows or is missing {sorted(missing)}.")
    (OUTPUT / f"{stem}.csv").write_text(completed.stdout, encoding="utf-8")
    return frame

def show_and_export(figure, name):
    for extension in ("png", "svg"):
        figure.savefig(OUTPUT / f"{name}.{extension}", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(figure)

## 1. Station states with and without reheat

Each row is a model station. The bypass outlet is a separate flow branch. The default wet and dry cases use different throat areas, so their overall differences also include the geometry change. The change from mixer outlet to nozzle inlet within the wet case shows the effect of reheat directly.

In [ ]:
station_columns = ("station", "pt_pa", "tt_k", "flow_kg_s", "fuel_air_ratio")
dry = run_csv("dry", DRY_COMMAND, station_columns).set_index("station", verify_integrity=True)
wet = run_csv("wet", WET_COMMAND, station_columns).set_index("station", verify_integrity=True)
if set(dry.index) != set(wet.index):
    raise ValueError("Dry and wet results must report the same station names.")
wet = wet.reindex(dry.index)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.4), sharey=True, layout="constrained")
positions = list(range(len(dry)))
for axis, column, scale, label in (
    (axes[0], "pt_pa", 1e-3, "Total pressure (kPa)"),
    (axes[1], "tt_k", 1.0, "Total temperature (K)"),
):
    for frame, offset, marker, color, name in (
        (dry, -0.13, "o", DRY_COLOR, "Dry case"),
        (wet, 0.13, "s", WET_COLOR, "Wet case"),
    ):
        axis.scatter(frame[column] * scale, [index + offset for index in positions],
                     marker=marker, s=36, color=color, label=name, zorder=3)
    axis.set_xlabel(label)
    axis.grid(axis="y", alpha=0.25)
axes[0].set_yticks(positions, [name.replace("_", " ") for name in dry.index])
axes[0].invert_yaxis()
axes[1].legend(frameon=False, loc="lower right")
fig.suptitle("Steady-state conditions by station")
show_and_export(fig, "station_states")

reheat = wet.loc[["mixer_exit", "nozzle_inlet"], ["tt_k", "pt_pa", "flow_kg_s"]].copy()
reheat["pt_pa"] *= 1e-3
reheat.columns = ["Total temperature (K)", "Total pressure (kPa)", "Gas flow (kg/s)"]
display(reheat.round(2))

## 2. Controlled response

The built-in dry demo steps normalized throttle from 0.50 to 0.75 at 0.10 s. The CSV contains one row per station at every time. Select one station to get a single time series, and show rotor-speed changes relative to trim so that the response of both shafts is visible.

In [ ]:
trace_columns = (
    "time_s", "station", "lp_rpm", "hp_rpm", "tt_k", "main_fuel_kg_s",
    "augmentor_fuel_kg_s", "throat_m2",
)
trajectory = run_csv("transient", TRANSIENT_COMMAND, trace_columns)
trace = trajectory.loc[trajectory["station"] == TRACE_STATION].sort_values("time_s")
if trace.empty or trace["time_s"].duplicated().any():
    raise ValueError(f"Expected one row per time for station {TRACE_STATION!r}.")
time = trace["time_s"]
initial = trace.iloc[0]

fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.2), sharex=True, layout="constrained")
shaft, temperature, fuel, throat = axes.flat
shaft.plot(time, trace["lp_rpm"] - initial["lp_rpm"], color=DRY_COLOR, label="LP shaft")
shaft.plot(time, trace["hp_rpm"] - initial["hp_rpm"], color=WET_COLOR,
           linestyle="--", label="HP shaft")
shaft.set_ylabel("Speed change (rpm)")
temperature.plot(time, trace["tt_k"], color=DRY_COLOR)
temperature.set_ylabel("Total temperature (K)")
temperature.set_title(TRACE_STATION.replace("_", " ").capitalize())
fuel.plot(time, (trace["main_fuel_kg_s"] - initial["main_fuel_kg_s"]) * 1000,
          color=DRY_COLOR, label="Main")
fuel.plot(time, (trace["augmentor_fuel_kg_s"] - initial["augmentor_fuel_kg_s"]) * 1000,
          color=WET_COLOR, linestyle="--", label="Augmentor")
fuel.set_ylabel("Fuel-flow change (g/s)")
throat.plot(time, (trace["throat_m2"] - initial["throat_m2"]) * 10000, color=DRY_COLOR)
throat.set_ylabel("Throat-area change (cm²)")
for axis in axes.flat:
    axis.grid(alpha=0.24)
for axis in (fuel, throat):
    axis.set_xlabel("Time (s)")
shaft.legend(frameon=False)
fuel.legend(frameon=False)
fig.suptitle("Rotor and actuator response")
show_and_export(fig, "transient_response")

## 3. Comparison with public PW-100 observations

The chart compares selected NASA flight markers and historical deck values with the simulator's synthetic EPR proxy. The error bars on the flight markers show estimated graph-reading uncertainty, rather than instrumentation uncertainty or a model prediction interval. See the [source and method](../docs/pw100-public-comparison.md).

In [ ]:
columns = ("mach", "flight_epr", "deck_epr", "surrogate_epr_proxy", "proxy_minus_flight_epr")
comparison = run_csv("comparison", ("compare-pw100",), columns).sort_values("mach")

fig, (epr, residual) = plt.subplots(
    2, 1, figsize=(8.6, 6.2), sharex=True, layout="constrained",
    gridspec_kw={"height_ratios": (2.4, 1.0)},
)
mach = comparison["mach"]
epr.errorbar(mach, comparison["flight_epr"], xerr=0.01, yerr=0.04,
             fmt="o", capsize=3, color=DRY_COLOR, label="NASA flight points")
epr.plot(mach, comparison["deck_epr"], color=DECK_COLOR, linestyle=":",
         marker="^", label="Historical deck")
epr.plot(mach, comparison["surrogate_epr_proxy"], color=WET_COLOR, linestyle="--",
         marker="s", label="Synthetic EPR proxy")
epr.set_ylabel("Engine pressure ratio")
epr.legend(frameon=False)
residual.axhline(0, color=DECK_COLOR, linewidth=1)
residual.vlines(mach, 0, comparison["proxy_minus_flight_epr"], color=WET_COLOR, linewidth=1.2)
residual.scatter(mach, comparison["proxy_minus_flight_epr"], color=WET_COLOR, s=26)
residual.set_ylabel("Proxy − flight EPR")
residual.set_xlabel("Flight Mach number")
for axis in (epr, residual):
    axis.grid(alpha=0.24)
fig.suptitle("PW-100 public comparison at 40,000 ft")
show_and_export(fig, "pw100_epr_comparison")

print(f"CSV, logs, and presentation figures saved to {OUTPUT}")